## Test generate pMHC functionality
This notebook is used to test changes tools/generate_pMHC_SCT.py script.
Use this to ensure proper function.

In [1]:
import sys
sys.path.insert(0, "..")
sys.path.insert(0, "../tools")

import pandas as pd
from config import ALL_DATA_DIR

In [2]:
import argparse
import sys
from pathlib import Path

import generate_pMHC_SCT as sct
from SCT_constants import (B2M_aa, 
                           HLA_aa, 
                           destination_vector_nuc,
                           leader_peptide_aa, 
                           linker_aa)


        None -> ValueError
          '' -> ValueError
       '   ' -> ValueError
 'pHIV_EGFP' -> KeyError
     'pUC19' -> KeyError
 'pHIV-EGFP' -> accepted, arms ['3_addon', '5_addon']
    --vector -> argparse exits when it is missing


In [3]:
input_csv = ALL_DATA_DIR / "HLA-A2_positive_SCTs.csv"

### Build SCT

In [4]:
destination_vector_nuc["pHIV-dTomato"]

{'5_addon': 'gaactgaattcatcgacgtt',
 '3_addon': 'aactattctagagtacccgggctaggatcc'}

In [5]:
sct.generate_sct_constructs(
    input_csv=input_csv,
    seed=4534,
    vector="pHIV-dTomato",
    hla="HLA-A201_RS",
    leader="HGH"
)

renamed column 'Name' -> 'name'
renamed column 'Epitope' -> 'peptide_aa'
seed used: 4534
seed used: 4534
seed used: 4534
seed used: 4534
seed used: 4534
seed used: 4534
seed used: 4534


,name,peptide_aa,peptide_nuc,to_order,hla,hla_sequence,leader,leader_sequence,vector,seed,GC_fraction
0,gp100,YLEPGPVTA,TACTTAGAGCCCGGCCCCGTGACAGCT,gaactgaattcatcgacgttgccaccATGGCCACCGGCAGCAGGAC...,HLA-A201_RS,GGCAGCCACAGCATGAGGTACTTCTTCACCAGCGTTTCTAGGCCCG...,HGH,ATGGCCACCGGCAGCAGGACCAGCCTGCTTCTGGCTTTTGGCTTGC...,pHIV-dTomato,4534,0.613
1,MART-1,ELAGIGILTV,GAGCTGGCCGGCATCGGCATCCTGACAGTT,gaactgaattcatcgacgttgccaccATGGCCACCGGCAGCAGGAC...,HLA-A201_RS,GGCAGCCACAGCATGAGGTACTTCTTCACCAGCGTTTCTAGGCCCG...,HGH,ATGGCCACCGGCAGCAGGACCAGCCTGCTTCTGGCTTTTGGCTTGC...,pHIV-dTomato,4534,0.613


### Validate generated sequences translates back into correct amino acids

In [6]:
from Bio.Seq import Seq

# convert nucleotide sequence to amino acids
HLA_RS = "GGCAGCCACAGCATGAGGTACTTCTTCACCAGCGTGTCTAGGCCTGGCAGGGGCGAGCCTAGGTTTATTGCCGTGGGATACGTGGACGACACCCAGTTCGTGAGGTTCGACAGCGATGCCGCTAGCCAAAGGATGGAGCCCAGGGCTCCTTGGATCGAGCAAGAGGGCCCCGAATATTGGGATGGCGAGACCAGAAAGGTGAAGGCCCACAGCCAGACCCACAGGGTGGATCTGGGCACATTAAGGGGAGCCTACAACCAGAGCGAGGCCGGCTCTCACACCGTGCAAAGGATGTACGGATGTGACGTGGGCAGCGACTGGAGATTCCTGAGGGGCTACCACCAGTACGCCTACGACGGCAAGGACTACATCGCCCTGAAGGAGGACCTTAGGAGCTGGACAGCTGCCGATATGGCCGCTCAGACCACCAAGCACAAGTGGGAAGCCGCTCACGTTGCTGAGCAGCTTAGGGCCTACCTTGAGGGCACCTGCGTGGAATGGCTGAGGAGGTACCTGGAGAACGGCAAGGAGACCCTGCAGAGGACCGATGCTCCCAAGACCCACATGACCCACCACGCTGTGAGCGACCATGAGGCCACCCTTAGGTGCTGGGCTTTGAGCTTCTACCCCGCCGAAATTACCCTGACCTGGCAGAGGGACGGCGAAGATCAGACCCAGGACACCGAGCTTGTGGAGACCAGACCCGCTGGAGATGGCACCTTCCAAAAGTGGGCTGCCGTGGTGGTTCCCAGCGGCCAAGAGCAGAGGTACACATGTCACGTGCAGCACGAGGGACTGCCCAAACCTCTGACCCTTAGGTGGGAGCCCAGCTCTCAGCCCACCATCCCCATTGTGGGCATTATTGCCGGCCTGGTTCTTTTTGGAGCCGTTATTACCGGCGCCGTTGTGGCCGCCGTGATGTGGAGGAGGAAGAGCAGCTGA"
HLA_RS_aa = Seq(HLA_RS).translate()
HLA_RS_aa == HLA_aa["HLA-A201_RS"]

True

In [7]:
# if different use this to find different areas
# comparisons between sequences
import difflib
ref = HLA_aa["HLA-A201_RS"]
qry = HLA_RS_aa
sm = difflib.SequenceMatcher(None, ref, qry, autojunk=False)
for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag != "equal":
        print(f"{tag:7s} ref[{i1}:{i2}]={ref[i1:i2]!r}  qry[{j1}:{j2}]={qry[j1:j2]!r}")
    else:
        print("Both strings are equal!")

Both strings are equal!


### Validating the add ons
Use benchling to validate if these sequences are the same. If not change the cells below to find mismatched sequences.

In [9]:
gp100_5 = "gaactgaattcatcgacgtt"
tomato_5 = "gctgaactgaattcatcgacgtt"

In [10]:
# if different use this to find different areas
# comparisons between sequences
import difflib
ref = tomato_5
qry = gp100_5
sm = difflib.SequenceMatcher(None, ref, qry, autojunk=False)
for tag, i1, i2, j1, j2 in sm.get_opcodes():
    if tag != "equal":
        print(f"{tag:7s} ref[{i1}:{i2}]={ref[i1:i2]!r}  qry[{j1}:{j2}]={qry[j1:j2]!r}")
    else:
        print("Both strings are equal!")

delete  ref[0:3]='gct'  qry[0:0]=''
Both strings are equal!
